# Part 3 – Tree Models, Ensembles, Tuning, and Serialization

In [ ]:

import os, joblib, numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, roc_auc_score

os.makedirs("outputs", exist_ok=True)

# Load data
possible_paths = [
    "data/cleaned_data.csv",
    "cleaned_data.csv",
    "https://raw.githubusercontent.com/NarutoxMessi/Capstone-Project/main/data/cleaned_data.csv",
    "https://raw.githubusercontent.com/NarutoxMessi/Capstone-Project/data/cleaned_data.csv"
]
df = None
for p in possible_paths:
    try:
        df = pd.read_csv(p)
        print(f"Loaded dataset from: {p}")
        break
    except Exception:
        pass
if df is None:
    raise FileNotFoundError("cleaned_data.csv not found locally or on GitHub.")

# Same preprocessing as Part 2
y_clf = (df["close"] > df["close"].median()).astype(int)
X = df.drop(columns=["close", "adj_close", "date"]).copy()
if "volume_category" in X.columns:
    X["volume_category"] = X["volume_category"].map({"Low":0,"Medium":1,"High":2})
categorical_nominal = [c for c in ["ticker","day_name"] if c in X.columns]
X = pd.get_dummies(X, columns=categorical_nominal, drop_first=True)

X_train, X_test, y_train, y_test = train_test_split(X, y_clf, test_size=0.2, random_state=42)
scaler = StandardScaler()
X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train), columns=X_train.columns, index=X_train.index)
X_test_scaled = pd.DataFrame(scaler.transform(X_test), columns=X_test.columns, index=X_test.index)

results = []

# Logistic from Part 2 for CV/test AUC
log_reg = LogisticRegression(max_iter=1000, random_state=42)
log_reg.fit(X_train_scaled, y_train)
log_test_auc = roc_auc_score(y_test, log_reg.predict_proba(X_test_scaled)[:,1])

# 1) Decision tree baseline
dt_base = DecisionTreeClassifier(random_state=42)
dt_base.fit(X_train_scaled, y_train)
dt_base_train_acc = accuracy_score(y_train, dt_base.predict(X_train_scaled))
dt_base_test_acc = accuracy_score(y_test, dt_base.predict(X_test_scaled))

# 2) Controlled decision tree
dt_ctrl = DecisionTreeClassifier(max_depth=5, min_samples_split=20, random_state=42)
dt_ctrl.fit(X_train_scaled, y_train)
dt_ctrl_train_acc = accuracy_score(y_train, dt_ctrl.predict(X_train_scaled))
dt_ctrl_test_acc = accuracy_score(y_test, dt_ctrl.predict(X_test_scaled))

# 3) Gini vs entropy
dt_gini = DecisionTreeClassifier(max_depth=5, criterion='gini', random_state=42).fit(X_train_scaled, y_train)
dt_entropy = DecisionTreeClassifier(max_depth=5, criterion='entropy', random_state=42).fit(X_train_scaled, y_train)
gini_test_acc = accuracy_score(y_test, dt_gini.predict(X_test_scaled))
entropy_test_acc = accuracy_score(y_test, dt_entropy.predict(X_test_scaled))

# 4) Random forest
rf = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42)
rf.fit(X_train_scaled, y_train)
rf_train_acc = accuracy_score(y_train, rf.predict(X_train_scaled))
rf_test_acc = accuracy_score(y_test, rf.predict(X_test_scaled))
rf_test_auc = roc_auc_score(y_test, rf.predict_proba(X_test_scaled)[:,1])

feat_imp = pd.DataFrame({"Feature": X_train.columns, "Importance": rf.feature_importances_}).sort_values("Importance", ascending=False)
top5 = feat_imp.head(5)
low5 = feat_imp.tail(5)
top5.to_csv("outputs/random_forest_top5_feature_importance.csv", index=False)

# 4a) Gradient boosting
gb = GradientBoostingClassifier(n_estimators=100, learning_rate=0.1, max_depth=3, random_state=42)
gb.fit(X_train_scaled, y_train)
gb_train_acc = accuracy_score(y_train, gb.predict(X_train_scaled))
gb_test_acc = accuracy_score(y_test, gb.predict(X_test_scaled))
gb_test_auc = roc_auc_score(y_test, gb.predict_proba(X_test_scaled)[:,1])

# 4b) Feature ablation
low5_features = low5["Feature"].tolist()
X_train_red = X_train_scaled.drop(columns=low5_features)
X_test_red = X_test_scaled.drop(columns=low5_features)
rf_reduced = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42)
rf_reduced.fit(X_train_red, y_train)
rf_reduced_auc = roc_auc_score(y_test, rf_reduced.predict_proba(X_test_red)[:,1])

# 5) Cross-validated comparison
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),
    "Decision Tree (max_depth=5)": DecisionTreeClassifier(max_depth=5, min_samples_split=20, random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42),
    "Gradient Boosting": GradientBoostingClassifier(n_estimators=100, learning_rate=0.1, max_depth=3, random_state=42)
}
cv_rows=[]
for name, model in cv_models.items():
    scores = cross_val_score(model, X_train_scaled, y_train, cv=cv, scoring='roc_auc')
    test_auc = {
        "Logistic Regression": log_test_auc,
        "Decision Tree (max_depth=5)": roc_auc_score(y_test, dt_ctrl.predict_proba(X_test_scaled)[:,1]),
        "Random Forest": rf_test_auc,
        "Gradient Boosting": gb_test_auc
    }[name]
    cv_rows.append([name, scores.mean(), scores.std(), test_auc])
cv_df = pd.DataFrame(cv_rows, columns=["Model","CV Mean AUC","CV Std AUC","Test AUC"])
cv_df.to_csv("outputs/part3_model_comparison.csv", index=False)

# 6) Grid search
pipeline = make_pipeline(SimpleImputer(strategy='median'), StandardScaler(), RandomForestClassifier(random_state=42))
param_grid = {
    'randomforestclassifier__n_estimators': [50, 100, 200],
    'randomforestclassifier__max_depth': [5, 10, None],
    'randomforestclassifier__min_samples_leaf': [1, 5]
}
grid = GridSearchCV(pipeline, param_grid, cv=cv, scoring='roc_auc', n_jobs=-1)
grid.fit(X_train, y_train)
best_pipeline = grid.best_estimator_
best_params = grid.best_params_
best_score = grid.best_score_

# 7) Manual learning curve
fractions = [0.2, 0.4, 0.6, 0.8, 1.0]
learning_rows=[]
for f in fractions:
    n = int(f * len(X_train))
    X_sub = X_train.iloc[:n]
    y_sub = y_train.iloc[:n]
    best_pipeline.fit(X_sub, y_sub)
    train_auc = roc_auc_score(y_sub, best_pipeline.predict_proba(X_sub)[:,1])
    test_auc = roc_auc_score(y_test, best_pipeline.predict_proba(X_test)[:,1])
    learning_rows.append([f, train_auc, test_auc])
learning_df = pd.DataFrame(learning_rows, columns=["Training fraction","Training AUC","Test AUC"])
learning_df.to_csv("outputs/manual_learning_curve.csv", index=False)

# 8) Serialize and reload
joblib.dump(best_pipeline, "best_model.pkl")
loaded_model = joblib.load("best_model.pkl")
sample_rows = X_test.head(2)
sample_preds = loaded_model.predict(sample_rows)
print("Reloaded model predictions on 2 handcrafted rows:", sample_preds)

# Print key outputs
print("\nDecision Tree baseline train/test acc:", dt_base_train_acc, dt_base_test_acc)
print("Controlled Decision Tree train/test acc:", dt_ctrl_train_acc, dt_ctrl_test_acc)
print("Gini vs Entropy test acc:", gini_test_acc, entropy_test_acc)
print("Random Forest train/test acc/AUC:", rf_train_acc, rf_test_acc, rf_test_auc)
print("\nTop 5 RF features:\n", top5)
print("\nGradient Boosting train/test acc/AUC:", gb_train_acc, gb_test_acc, gb_test_auc)
print("\nFeature ablation AUC full vs reduced:", rf_test_auc, rf_reduced_auc)
print("\nCV comparison:\n", cv_df)
print("\nBest GridSearch params:", best_params)
print("Best GridSearch ROC-AUC:", best_score)
print("\nManual learning curve:\n", learning_df)
